# Binance candle backfill and inspection

This notebook uses the same backfill function as the command-line collector. It is safe to rerun because existing candles are upserted, not duplicated. It never places trades.

In [2]:
from pathlib import Path
import pandas as pd
from crypto_trader.data.binance import backfill_historical_candles, open_candle_database

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
database_path = project_root / "data" / "database" / "trading.db"
symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "DOGEUSDT"]
backfill_days = 30
connection = open_candle_database(database_path)

## Backfill

Change `symbols` or `backfill_days` above, then run this cell. A 30-day one-minute backfill is 43,200 rows per symbol and may take several minutes.

In [3]:
processed_rows = backfill_historical_candles(
    connection,
    symbols=symbols,
    minutes=backfill_days * 24 * 60,
)
print(f"Processed {processed_rows:,} candle rows.")

Processed 216,000 candle rows.


## Inspect coverage and schema

The following cells show how many rows exist, the time range for each symbol, recent prices, and the actual SQLite schema.

In [4]:
coverage = pd.read_sql_query(
    """
    SELECT exchange, symbol, interval_seconds, COUNT(*) AS rows,
           MIN(open_time_ms) AS first_open_time_ms,
           MAX(open_time_ms) AS last_open_time_ms
    FROM candles
    GROUP BY exchange, symbol, interval_seconds
    ORDER BY symbol
    """,
    connection,
)
coverage

,exchange,symbol,interval_seconds,rows,first_open_time_ms,last_open_time_ms
0,binance,BTCUSDT,60,43200,1788906720000,1791498660000
1,binance,DOGEUSDT,60,43200,1788906720000,1791498660000
2,binance,ETHUSDT,60,43200,1788906720000,1791498660000
3,binance,SOLUSDT,60,43200,1788906720000,1791498660000
4,binance,XRPUSDT,60,43200,1788906720000,1791498660000


In [5]:
latest = pd.read_sql_query(
    """
    SELECT symbol, datetime(open_time_ms / 1000, 'unixepoch') AS open_time_utc,
           open, high, low, close, volume
    FROM candles
    WHERE interval_seconds = 60
    ORDER BY open_time_ms DESC
    LIMIT 25
    """,
    connection,
)
latest

,symbol,open_time_utc,open,high,low,close,volume
0,DOGEUSDT,2026-10-08 22:31:00,0.08435,0.08435,0.08432,0.08432,69533.00000
1,XRPUSDT,2026-10-08 22:31:00,1.37960,1.37980,1.37940,1.37950,4787.90000
2,SOLUSDT,2026-10-08 22:31:00,110.20000,110.21000,110.12000,110.14000,1016.91600
3,ETHUSDT,2026-10-08 22:31:00,2473.60000,2473.79000,2472.93000,2473.16000,48.19830
4,BTCUSDT,2026-10-08 22:31:00,81787.75000,81787.75000,81770.00000,81770.01000,1.79231
5,DOGEUSDT,2026-10-08 22:30:00,0.08436,0.08436,0.08435,0.08435,23425.00000
6,XRPUSDT,2026-10-08 22:30:00,1.38010,1.38010,1.37960,1.37970,30405.90000
7,SOLUSDT,2026-10-08 22:30:00,110.23000,110.25000,110.20000,110.20000,1178.54200
8,ETHUSDT,2026-10-08 22:30:00,2474.21000,2474.22000,2473.60000,2473.60000,64.76000
9,BTCUSDT,2026-10-08 22:30:00,81787.75000,81787.75000,81787.74000,81787.74000,2.85656


In [6]:
schema = pd.read_sql_query(
    "SELECT type, name, sql FROM sqlite_master WHERE sql IS NOT NULL ORDER BY type, name",
    connection,
)
schema

,type,name,sql
0,index,idx_candles_interval_open_time,CREATE INDEX idx_candles_interval_open_time\nO...
1,table,candles,CREATE TABLE candles (\n -- Internal row id...


In [7]:
# Close the connection when finished so other services can use the database.
connection.close()